# 12. Mini capstone — eksperimen end-to-end dengan audit validitas

**Capaian:** Membangun baseline, perbaikan model, analisis kegagalan, dan pelaporan ilmiah.

**Prasyarat:** Python dasar, NumPy, dan membaca array. **Estimasi:** 75–100 menit.

> Kerjakan dari atas ke bawah. Setiap grafik wajib disertai penafsiran, bukan hanya ditempel.

## Konsep dan model

Gunakan kumpulan tablet sintetis sebagai *benchmark demonstrasi*. Pipeline: masalah → protokol akuisisi → preprocessing → fitur → split berdasarkan batch → baseline → model → evaluasi → ablation → rencana validasi nyata. Test tidak boleh ikut dipakai untuk pemilihan hiperparameter berulang.

## Bangun dataset dan daftar metadata

In [ ]:
import numpy as np
import pandas as pd
from pengolahan_citra_id.datasets import tablet_sintetis
from pengolahan_citra_id.features import fitur_tablet,FITUR_TABLET
from pengolahan_citra_id.evaluation import split_berdasar_batch
images,y,group=tablet_sintetis(16,12,seed=12)
X=np.stack([fitur_tablet(im) for im in images])
meta=pd.DataFrame({'label':y,'batch':group})
print(meta.groupby('batch')['label'].agg(['size','mean']).head())
tr,te=split_berdasar_batch(y,group,seed=16)
print('Batch train:',sorted(set(group[tr])),'\nBatch test:',sorted(set(group[te])))

## Baseline vs model uji

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
baseline=DummyClassifier(strategy='most_frequent').fit(X[tr],y[tr])
model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,random_state=42)).fit(X[tr],y[tr])
for nama,est in [('Mayoritas',baseline),('Regresi logistik',model)]:
    print('\n',nama,'\n',classification_report(y[te],est.predict(X[te]),zero_division=0))

## Ablasi satu fitur tanpa retrain pada test

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_score
cv=GroupKFold(n_splits=4)
for label,cols in [('Semua',list(range(X.shape[1]))),('Geometri saja',list(range(4))),('Intensitas saja',[4,5,6])]:
    estimator=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,random_state=42))
    scores=cross_val_score(estimator,X[tr][:,cols],y[tr],groups=group[tr],cv=cv,scoring='f1_macro')
    print(label,'CV train F1=',scores.round(3),'rerata=',round(scores.mean(),3))

## Interpretasi keterbatasan

In [ ]:
import matplotlib.pyplot as plt
coef=model.named_steps['logisticregression'].coef_[0]
order=np.argsort(np.abs(coef))[::-1]
fig,ax=plt.subplots(figsize=(8,3))
ax.bar(np.array(FITUR_TABLET)[order],coef[order]);ax.axhline(0,color='black',linewidth=0.7)
ax.tick_params(axis='x',rotation=50);ax.set_ylabel('Koefisien pada fitur standar');plt.tight_layout();plt.show()

## Diskusi, HOTS, dan latihan

**Tugas akhir:** buat laporan 6–8 halaman: rumusan masalah, jenis data & keterbatasan, diagram pipeline, baseline, tabel metrik, analisis per batch, confusion matrix, ablasi, rencana kalibrasi, rencana validasi lapangan, dan 5 daftar pustaka.

**Pertanyaan lanjut:** bagaimana Anda mendeteksi domain shift pada kamera baru? Apa metode untuk memperkirakan ketidakpastian model? Kapan accuracy tidak cukup?

**Laporan singkat:** rumusan masalah → rancangan eksperimen → hasil → interpretasi → keterbatasan → kesimpulan.